# Performance Evaluation

Load a trained PPO checkpoint and run detailed evaluation: per-episode stats, score distributions, and AI vs Human comparison.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from src.agent.ppo_agent import PPOAgent
from src.env.car_env import make_env
from src.evaluate.evaluate import evaluate_with_details
from src.utils import load_env_config, load_train_config

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})

CHECKPOINT = "../models/ppo_best/best_model.pt"
N_EPISODES = 30

: 

In [ ]:
env_cfg = load_env_config("../configs/env_config.yaml")
train_cfg = load_train_config("../configs/train_config.yaml")

tmp = make_env(env_cfg, seed=0)
n_actions = tmp.action_space.n
tmp.close()

agent = PPOAgent(train_cfg, env_cfg, n_actions)
agent.load(CHECKPOINT)
print("Model loaded successfully")

In [ ]:
results = evaluate_with_details(agent, env_cfg, n_episodes=N_EPISODES, deterministic=True)

print(f"Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
print(f"Min / Max:   {results['min_reward']:.1f} / {results['max_reward']:.1f}")
print(f"Mean steps:  {results['mean_steps']:.0f}")

In [ ]:
rewards = [e["reward"] for e in results["episodes"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(rewards)), rewards, color="steelblue", alpha=0.8)
axes[0].axhline(results["mean_reward"], color="red", linestyle="--", label="Mean")
axes[0].axhline(800, color="green", linestyle=":", label="Human baseline")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward")
axes[0].set_title("Per-Episode Rewards")
axes[0].legend()

axes[1].hist(rewards, bins=15, color="coral", edgecolor="black", alpha=0.75)
axes[1].axvline(800, color="green", linestyle=":", label="Human baseline")
axes[1].set_xlabel("Reward")
axes[1].set_ylabel("Count")
axes[1].set_title("Reward Distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig("../outputs/plots/performance_eval.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
human_mean = 800
ai_mean = results["mean_reward"]

labels = ["Human", "PPO Agent"]
means = [human_mean, ai_mean]
colors = ["#4FC3F7", "#FF8A65"]

plt.figure(figsize=(6, 5))
bars = plt.bar(labels, means, color=colors, edgecolor="black", width=0.5)
plt.ylabel("Mean Reward")
plt.title("Human vs AI Performance")
for bar, val in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val:.0f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"AI vs Human delta: {ai_mean - human_mean:+.1f}")
print(f"AI {'beats' if ai_mean > human_mean else 'below'} human baseline")